In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install -q \
    fastapi \
    uvicorn \
    pyngrok \
    nest-asyncio \
    faster-whisper \
    transformers \
    accelerate \
    torch \
    python-multipart \
    requests

# Forcefully remove the deprecated parameter directly from the installed package
!sed -i 's/, metadata_errors="ignore"//g' /usr/local/lib/python3.13/dist-packages/faster_whisper/audio.py

In [3]:
import os
import torch
from kaggle_secrets import UserSecretsClient
from faster_whisper import WhisperModel
from transformers import AutoTokenizer, AutoModelForCausalLM
from pyngrok import ngrok

# Authenticate tokens
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
ngrok_token = user_secrets.get_secret("NGROK_AUTHTOKEN")

ngrok.set_auth_token(ngrok_token)

# 1. Initialize Whisper STT (CTranslate2 on CUDA)
print("Loading Whisper Large-v3-Turbo...")
stt_model = WhisperModel("large-v3-turbo", device="cuda", compute_type="float16")

# 2. Initialize Gemma-2-2B-IT
print("Loading Gemma-2-2b-it...")
model_id = "google/gemma-2-2b-it"
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
llm_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
    token=hf_token
)
print("Both models loaded successfully into VRAM.")

Loading Whisper Large-v3-Turbo...
Loading Gemma-2-2b-it...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Both models loaded successfully into VRAM.


In [4]:
import re
import json
import tempfile
import subprocess
import os
import requests
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

app = FastAPI(title="Air-Gapped AI Intake Engine")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

FIELD_TYPE_MAP = {
    0: "short_text", 1: "paragraph", 2: "radio",
    3: "dropdown",   4: "checkbox",  5: "linear_scale",
    7: "grid",       9: "date",     10: "time", 13: "file_upload",
}

def extract_form_fields(google_form_url: str):
    """Scrapes entry IDs, labels, types, and options from a public Google Form."""
    google_form_url = google_form_url.replace("/edit", "/viewform")
    try:
        res = requests.get(google_form_url, timeout=10)
        match = re.search(r'var FB_PUBLIC_LOAD_DATA_ = (\[.*?\]);\n', res.text, re.DOTALL)
        if not match:
            match = re.search(r'FB_PUBLIC_LOAD_DATA_\s*=\s*(\[.*?\]);', res.text, re.DOTALL)
        if not match:
            raise ValueError("Could not find form metadata.")

        raw_data = json.loads(match.group(1))
        questions = raw_data[1][1]
        fields = []
        for q in questions:
            if q and len(q) > 4 and q[4]:
                label = q[1]
                type_code = q[3]
                field_type = FIELD_TYPE_MAP.get(type_code, "short_text")
                entry_id = f"entry.{q[4][0][0]}"
                field_info = {"id": entry_id, "label": label, "type": field_type}
                
                # Extract options for choice-based fields safely
                if type_code in [2, 3, 4] and len(q[4][0]) > 1 and q[4][0][1]:
                    field_info["options"] = [opt[0] for opt in q[4][0][1] if opt and opt[0]]
                
                fields.append(field_info)
        return fields
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Failed to scrape form: {str(e)}")

def run_gemma_mapping(transcript: str, fields: list) -> dict:
    """Prompts Gemma-2-2B-IT to map transcript data into form entry IDs with correct formats."""
    fields_str = json.dumps(fields, indent=2)
    prompt = f"""<start_of_turn>user
You are an accurate automated data-entry assistant.
Task: Map the spoken transcript into the provided form fields.

Target Form Fields:
{fields_str}

Spoken Transcript:
"{transcript}"

Requirements:
1. Output ONLY a valid JSON dictionary where each key is the exact field 'id' (e.g., "entry.123456789").
2. Format values by field type:
   - "short_text" / "paragraph": plain text string
   - "date": output as "YYYY-MM-DD" (e.g., "2024-10-13")
   - "time": output as "HH:MM" in 24-hour format (e.g., "14:30" for 2:30 PM)
   - "radio" / "dropdown": output the EXACT matching option text from the options list
   - "checkbox": output comma-separated matching option texts (e.g., "Option A, Option C")
   - "linear_scale": output a single number string (e.g., "3")
3. If a field was not mentioned, omit it or set its value to "".
4. Do not include markdown code blocks. Return the raw JSON object only.<end_of_turn>
<start_of_turn>model
"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs, 
            max_new_tokens=512, 
            temperature=0.1, 
            do_sample=False
        )
    response_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    
    json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
    if not json_match:
        return {}
    return json.loads(json_match.group(0))

class ScrapeRequest(BaseModel):
    url: str

@app.post("/api/v1/parse-fields")
async def parse_fields_endpoint(req: ScrapeRequest):
    fields = extract_form_fields(req.url)
    return {"status": "success", "fields": fields}

@app.post("/api/v1/process-audio")
async def process_audio_endpoint(
    audio_file: UploadFile = File(...),
    fields_json: str = Form(...) 
):
    fields = json.loads(fields_json)
    audio_bytes = await audio_file.read()
    
    # 1. Guard against empty uploads (< 1KB)
    if len(audio_bytes) < 1000:
        raise HTTPException(status_code=400, detail="Audio file too short or empty. Speak for at least 2 seconds.")

    # 2. Write raw audio to temporary file
    with tempfile.NamedTemporaryFile(delete=False, suffix=".webm") as raw_tmp:
        raw_tmp.write(audio_bytes)
        raw_path = raw_tmp.name

    wav_path = raw_path.replace(".webm", ".wav")

    try:
        # 3. Transcode to valid 16kHz mono WAV via ffmpeg (prevents PyAV EOF crashes)
        cmd = [
            "ffmpeg", "-y", "-i", raw_path,
            "-ar", "16000", "-ac", "1", "-c:a", "pcm_s16le",
            wav_path
        ]
        result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        if result.returncode != 0:
            raise RuntimeError(f"FFmpeg transcode failed: {result.stderr.decode('utf-8', errors='ignore')}")

        # 4. Transcribe clean WAV with Whisper
        segments, _ = stt_model.transcribe(wav_path, beam_size=1)
        transcript = " ".join([seg.text for seg in segments]).strip()

        if not transcript:
            raise HTTPException(status_code=400, detail="No speech detected in audio.")

        # 5. Map entries via Gemma
        mapped_data = run_gemma_mapping(transcript, fields)

        return {
            "status": "success",
            "transcript": transcript,
            "mapped_entries": mapped_data
        }
    finally:
        if os.path.exists(raw_path):
            os.remove(raw_path)
        if os.path.exists(wav_path):
            os.remove(wav_path)

In [ ]:
import nest_asyncio
import uvicorn
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient

# Authenticate ngrok
user_secrets = UserSecretsClient()
ngrok_token = user_secrets.get_secret("NGROK_AUTHTOKEN")
ngrok.set_auth_token(ngrok_token)

# Kill any lingering ngrok tunnels
ngrok.kill()

# Open tunnel on port 8000
public_tunnel = ngrok.connect(8000)
print("\n" + "="*60)
print(f"PUBLIC BACKEND API URL: {public_tunnel.public_url}")
print("="*60 + "\n")

# PATCH 3: Run Uvicorn asynchronously to prevent Jupyter event loop crashes
config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()


PUBLIC BACKEND API URL: https://rewrite-climatic-shredder.ngrok-free.dev



INFO:     Started server process [749]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     152.58.31.149:0 - "OPTIONS /api/v1/parse-fields HTTP/1.1" 200 OK
INFO:     152.58.31.149:0 - "POST /api/v1/parse-fields HTTP/1.1" 200 OK


[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


INFO:     152.58.31.149:0 - "POST /api/v1/process-audio HTTP/1.1" 200 OK
INFO:     152.58.31.149:0 - "POST /api/v1/process-audio HTTP/1.1" 200 OK
